# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tracy030115/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Since we don't have article text, we can't do actual clustering. Instead, we can sort pages into behavioral archetypes using explicit thresholds on the structured metrics available: how much search demand a page gets, how well it ranks, whether its clicks match that ranking, how engaged visitors are once they land, how stale the content is, and whether it's competing with another page from the same client for the same keyword. Each page gets exactly one archetype based on which combination of these signals it matches.

Reason codes (archetypes) it can output:

champions — good position, high demand, healthy CTR, not stale. Working as intended.

hidden_gems — good position and healthy CTR, but lower demand/volume. Underexposed, not underperforming.

rising_stars — not yet ranking well, but CTR is beating expectations for its position and has some real volume. Worth watching, possibly close to breaking out.

stale_visible_pages — good position, high demand, but hasn't been updated in a long time.

cannibalization_risk — same client has more than one page ranking for the same keyword.

engagement_problem_pages — good position, high demand, but low on-page engagement (GA4 side) despite search performing fine.

weak_no_demand_pages — very low impressions; little to no search demand at all.
monitor — doesn't clearly match any of the above; needs a human look rather than an automatic label.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    ),
    staleness AS (
        SELECT
            dc.content_hash_id,
            DATE_DIFF('day', dc.content_updated_date, ref.max_date) AS days_since_last_update
        FROM read_parquet('{rel}/dim_content.parquet') dc
        CROSS JOIN ref
    )
    SELECT
        CASE
            WHEN s.days_since_last_update <= 30 THEN '1_fresh_0-30d'
            WHEN s.days_since_last_update <= 180 THEN '2_moderate_31-180d'
            ELSE '3_stale_180d+'
        END AS staleness_bucket,
        COUNT(DISTINCT s.content_hash_id) AS n,
        AVG(f.gsc_clicks) AS avg_clicks,
        AVG(f.gsc_impressions) AS avg_impressions
    FROM staleness s
    JOIN read_parquet('{rel}/fact_content_daily_performance_sample.parquet') f
        ON s.content_hash_id = f.content_hash_id
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_clicks,avg_impressions
0,1_fresh_0-30d,142871,0.235757,37.176250
1,2_moderate_31-180d,258913,0.036821,9.195252
2,3_stale_180d+,7421,0.001768,0.425479


In [3]:
con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1_pos_1-3'
            WHEN gsc_avg_position <= 10 THEN '2_pos_4-10'
            WHEN gsc_avg_position <= 20 THEN '3_pos_11-20'
            ELSE '4_pos_21plus'
        END AS position_bucket,
        COUNT(*) AS n,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM (
        SELECT gsc_avg_position, gsc_clicks, gsc_impressions
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        WHERE gsc_impressions > 0
    )
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,total_clicks,total_impressions,ctr
0,1_pos_1-3,440501,387249.0,13927471.0,0.027805
1,2_pos_4-10,1530594,657613.0,155493934.0,0.004229
2,3_pos_11-20,633342,101416.0,24236904.0,0.004184
3,4_pos_21plus,1274500,62839.0,22536563.0,0.002788


Staleness: CONFIRMED

CTR-vs-position: MIXED

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

page_level = con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    ),
    fact_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position,
            SUM(ga4_sessions) AS total_sessions,
            SUM(ga4_engaged_sessions) AS total_engaged_sessions,
            BOOL_OR(gsc_data_available) AS gsc_data_available_any,
            BOOL_OR(ga4_data_available) AS ga4_data_available_any
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    content_meta AS (
        SELECT
            content_hash_id,
            keyword_hash_id,
            content_updated_date,
            is_published,
            is_deleted
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published = true AND is_deleted = false
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        cm.keyword_hash_id,
        f.total_impressions,
        f.total_clicks,
        f.avg_position,
        f.total_sessions,
        f.total_engaged_sessions,
        f.gsc_data_available_any,
        f.ga4_data_available_any,
        DATE_DIFF('day', cm.content_updated_date, ref.max_date) AS days_since_last_update,
        CASE WHEN f.total_impressions > 0 THEN f.total_clicks * 1.0 / f.total_impressions ELSE NULL END AS ctr,
        CASE WHEN f.total_sessions > 0 THEN f.total_engaged_sessions * 1.0 / f.total_sessions ELSE NULL END AS engagement_rate
    FROM fact_agg f
    JOIN content_meta cm ON f.content_hash_id = cm.content_hash_id
    CROSS JOIN ref
""").df()

print(page_level.shape)
page_level.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(394928, 13)


,client_hash_id,content_hash_id,keyword_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_engaged_sessions,gsc_data_available_any,ga4_data_available_any,days_since_last_update,ctr,engagement_rate
0,client_3ffa76342f366962,content_73f21e612565035a,keyword_41684cca996c95a7,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
1,client_3ffa76342f366962,content_5a5be514ff559598,keyword_66ba2c2dfa653eed,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
2,client_3ffa76342f366962,content_05b377d0c8a5cfd8,keyword_935dbe1165583ac2,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
3,client_3ffa76342f366962,content_dcbfbaf912c88c76,keyword_803bd30a78b2b2d2,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
4,client_3ffa76342f366962,content_44de3cf116b9e3f1,keyword_28987d446a1ac934,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN


In [11]:
def position_bucket(pos):
    if pos is None or pd.isna(pos):
        return None
    if pos <= 3:
        return "1_pos_1-3"
    elif pos <= 10:
        return "2_pos_4-10"
    elif pos <= 20:
        return "3_pos_11-20"
    else:
        return "4_pos_21plus"

# expected CTR per bucket, taken directly from the verified bucket table
expected_ctr_by_bucket = {
    "1_pos_1-3": 0.027805,
    "2_pos_4-10": 0.004229,
    "3_pos_11-20": 0.004184,
    "4_pos_21plus": 0.002788,
}

page_level["position_bucket"] = page_level["avg_position"].apply(position_bucket)
page_level["expected_ctr"] = page_level["position_bucket"].map(expected_ctr_by_bucket)
page_level["ctr_gap"] = page_level["expected_ctr"] - page_level["ctr"]

In [12]:
# cannibalization: same client, same keyword, more than one page ranking
keyword_counts = (
    page_level[page_level["total_impressions"] > 0]
    .groupby(["client_hash_id", "keyword_hash_id"])["content_hash_id"]
    .nunique()
    .reset_index(name="pages_ranking_for_keyword")
)
page_level = page_level.merge(keyword_counts, on=["client_hash_id", "keyword_hash_id"], how="left")
page_level["cannibalization_risk"] = page_level["pages_ranking_for_keyword"].fillna(0) > 1

In [23]:
MIN_IMPRESSIONS = 10
STALE_THRESHOLD_DAYS = 180
HIGH_VOLUME_IMPRESSIONS = page_level["total_impressions"].quantile(0.75)  # data-derived, not guessed

def assign_archetype(row):
    gsc_available = bool(row["gsc_data_available_any"]) if pd.notna(row["gsc_data_available_any"]) else False
    if (not gsc_available) or row["total_impressions"] < MIN_IMPRESSIONS:
        return "INSUFFICIENT_DATA"

    is_good_position = pd.notna(row["avg_position"]) and row["avg_position"] <= 10
    is_high_volume = row["total_impressions"] >= HIGH_VOLUME_IMPRESSIONS
    is_low_ctr = pd.notna(row["ctr_gap"]) and row["ctr_gap"] > 0
    is_stale = pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] > STALE_THRESHOLD_DAYS
    has_low_engagement = pd.notna(row["engagement_rate"]) and row["engagement_rate"] < 0.3
    is_no_demand = row["total_impressions"] < MIN_IMPRESSIONS * 3
    ga4_available = bool(row["ga4_data_available_any"]) if pd.notna(row["ga4_data_available_any"]) else False
    is_cannibalization = bool(row["cannibalization_risk"]) if pd.notna(row["cannibalization_risk"]) else False

    if is_cannibalization:
        return "cannibalization_risk"
    if is_good_position and is_high_volume and not is_low_ctr and not is_stale:
        return "champions"
    if is_good_position and is_high_volume and is_stale:
        return "stale_visible_pages"
    if ga4_available and is_good_position and is_high_volume and has_low_engagement:
        return "engagement_problem_pages"
    if is_good_position and not is_high_volume and not is_low_ctr:
        return "hidden_gems"
    if not is_good_position and pd.notna(row["ctr_gap"]) and row["ctr_gap"] < 0 and row["total_impressions"] >= HIGH_VOLUME_IMPRESSIONS * 0.25:
        return "rising_stars"
    if is_no_demand:
        return "weak_no_demand_pages"

    return "monitor"

page_level["archetype"] = page_level.apply(assign_archetype, axis=1)
page_level["archetype"].value_counts()

,count
archetype,
INSUFFICIENT_DATA,239517
monitor,73963
rising_stars,23249
weak_no_demand_pages,22024
champions,18536
engagement_problem_pages,14692
hidden_gems,2918
stale_visible_pages,27
cannibalization_risk,2


In [24]:
action_map = {
    "champions": "protect",
    "hidden_gems": "improve",
    "rising_stars": "monitor",
    "stale_visible_pages": "refresh",
    "engagement_problem_pages": "rewrite",
    "cannibalization_risk": "merge",
    "weak_no_demand_pages": "prune",
    "monitor": "monitor",
    "INSUFFICIENT_DATA": "no_action_insufficient_data",
}
page_level["recommended_action"] = page_level["archetype"].map(action_map)

In [25]:
output_cols = [
    "client_hash_id", "content_hash_id", "archetype", "recommended_action",
    "total_impressions", "total_clicks", "ctr", "avg_position",
    "engagement_rate", "days_since_last_update", "cannibalization_risk"
]

os.makedirs("work/outputs", exist_ok=True)
page_level[output_cols].to_csv("work/outputs/archetype_rule_based.csv", index=False)

print(f"Wrote {len(page_level)} rows to work/outputs/archetype_rule_based.csv")
page_level["archetype"].value_counts()

Wrote 394928 rows to work/outputs/archetype_rule_based.csv


,count
archetype,
INSUFFICIENT_DATA,239517
monitor,73963
rising_stars,23249
weak_no_demand_pages,22024
champions,18536
engagement_problem_pages,14692
hidden_gems,2918
stale_visible_pages,27
cannibalization_risk,2


In [26]:
# cluster profile — typical numbers per archetype
profile_cols = ["total_impressions", "total_clicks", "ctr", "avg_position", "engagement_rate", "days_since_last_update"]
cluster_profile = page_level.groupby("archetype").agg(
    n=("content_hash_id", "count"),
    **{f"avg_{c}": (c, "mean") for c in profile_cols}
).reset_index()
cluster_profile

,archetype,n,avg_total_impressions,avg_total_clicks,avg_ctr,avg_avg_position,avg_engagement_rate,avg_days_since_last_update
0,INSUFFICIENT_DATA,239517,0.612746,0.003361,0.006369,20.470839,0.034365,45.031355
1,cannibalization_risk,2,236.500000,1.000000,0.002309,30.936609,0.000000,41.000000
2,champions,18536,2626.056269,20.675982,0.009316,6.471670,0.043284,30.369605
3,engagement_problem_pages,14692,5638.626940,13.305336,0.002252,6.838177,0.022701,34.887490
4,hidden_gems,2918,55.881768,1.373886,0.035272,6.438959,0.049712,36.719328
5,monitor,73963,860.266850,1.412598,0.000622,27.730899,0.035044,36.196531
6,rising_stars,23249,740.962837,5.385221,0.009641,22.617230,0.046761,38.246634
7,stale_visible_pages,27,1324.518519,7.814815,0.007535,7.125651,0.048397,225.333333
8,weak_no_demand_pages,22024,17.921722,0.028287,0.001623,29.711530,0.025623,40.235516


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
review_archetypes = [
    "champions", "hidden_gems", "rising_stars", "stale_visible_pages",
    "engagement_problem_pages", "cannibalization_risk", "weak_no_demand_pages"
]

top_per_archetype = (
    page_level[page_level["archetype"].isin(review_archetypes)]
    .sort_values("total_impressions", ascending=False)
    .groupby("archetype")
    .head(3)  # adjust so total rows across archetypes ≈ 20
    .sort_values(["archetype", "total_impressions"], ascending=[True, False])
)

cols_to_show = [
    "archetype", "client_hash_id", "content_hash_id", "total_impressions",
    "total_clicks", "ctr", "avg_position", "engagement_rate", "days_since_last_update"
]

top_per_archetype[cols_to_show]

,archetype,client_hash_id,content_hash_id,total_impressions,total_clicks,ctr,avg_position,engagement_rate,days_since_last_update
283946,cannibalization_risk,client_0fa64a184f18a4a0,content_82c29289f238d88a,433.0,2.0,0.004619,4.798657,0.000000,41
283929,cannibalization_risk,client_0fa64a184f18a4a0,content_248a44c30a4bc707,40.0,0.0,0.000000,57.074561,NaN,41
32332,champions,client_06d356715a8ff3b6,content_f88878f155e4838d,277353.0,2543.0,0.009169,5.034392,0.000000,6
87242,champions,client_b77d0d5f08f05e64,content_ac1ddc0c0e79289f,181605.0,2863.0,0.015765,5.471849,0.024783,6
183951,champions,client_e5c2aa26a8598242,content_ef7013c86d07aa99,150344.0,692.0,0.004603,6.814726,0.008180,20
37208,engagement_problem_pages,client_e547b89c05043229,content_963de14b1f58978f,615012.0,1676.0,0.002725,6.397556,0.008458,8
35970,engagement_problem_pages,client_e547b89c05043229,content_eadb33b5df496f4a,591696.0,3817.0,0.006451,2.258612,0.073053,18
233569,engagement_problem_pages,client_e547b89c05043229,content_545bb6cc7081ded3,585712.0,3048.0,0.005204,2.110776,0.053912,18
217644,hidden_gems,client_b10cb2997d0c7c86,content_411aefc7089cc603,109.0,1.0,0.009174,8.362507,0.000000,125
373642,hidden_gems,client_157ffe4d4a595515,content_a570b9143d48745b,109.0,1.0,0.009174,5.494102,0.000000,41


### 1.
**Archetype / action:** cannibalization_risk → merge

**Why it landed here:** Same client has more than one page (this and #2 below) ranking for the same keyword.

**Confidence:** Moderate. 433 impressions is a real volume, and position 4.8 with only 2 clicks (CTR 0.46%, well under the ~2.8% expected at that position) suggests this page isn't winning the clicks its ranking should earn — possibly because a sibling page is splitting demand.

**What would make it wrong:** If the two pages target genuinely distinct search intents under the same broad keyword tag (e.g. one is a buying guide, the other a product page), this isn't cannibalization at all — it's a keyword-tagging artifact, and merging would destroy a legitimately separate page.

### 2.
**Archetype / action:** cannibalization_risk → merge

**Why it landed here:** The sibling page to #1, same client/keyword.

**Confidence:** Low on its own. Position 57 with 0 clicks on 40 impressions, meaning this page is barely visible. The real evidence for cannibalization is #1's underperformance, not this page's numbers directly.

**What would make it wrong:** If this page is simply a weak, aging duplicate that should be pruned outright rather than merged, treating both as a "merge" pair may be the wrong action — this one might be closer to prune, with #1 kept as-is.

### 3.
**Archetype / action:** champions → protect

**Why it landed here:** High volume (277,353 impressions), good position (5.0), CTR (0.92%) at or near expected for that band, not stale (6 days).

**Confidence:** High. Very large impression base makes this a well-evidenced pattern, not noise.

**What would make it wrong:** `engagement_rate = 0.0` despite strong search performance is odd for a genuine "champion", It's worth checking whether GA4 tracking is actually firing on this page, since a true champion should show some on-page engagement, not none.

### 4.
**Archetype / action:** champions → protect

**Why it landed here:** High volume (181,605), good position (5.5), healthy CTR (1.58%, above expected), fresh (6 days).

**Confidence:** High. Strong volume and a CTR that beats the position baseline is a genuinely strong signal.

**What would make it wrong:** This is close to the ideal profile for the archetype. Worth double-checking only that the 6-day freshness isn't itself an artifact of the reporting-lag issue discussed earlier.

### 5.
**Archetype / action:** champions → protect

**Why it landed here:** High volume (150,344), good position (6.8), CTR (0.46%) roughly at expected for that band, not stale (20 days).

**Confidence:** Moderate-high. Volume is strong, though CTR sits right at the boundary of "expected," not clearly beating it — a weaker champion than #3/#4.

**What would make it wrong:** If the expected-CTR baseline for position 4–10 (0.42%) is itself noisy (recall the earlier MIXED verdict, CTR is nearly flat across positions 4–20), "meets expected CTR" is a low bar here, so this page's CTR performance is less impressive than the label implies.

### 6.
**Archetype / action:** engagement_problem_pages → rewrite

**Why it landed here:** Very high volume (615,012 impressions), good position (6.4), but engagement_rate only 0.85%.

**Confidence:** High on volume, moderate on the engagement interpretation. The traffic numbers are unambiguous; whether low engagement means "bad content" versus "quick-answer content that doesn't need engagement" is a judgment call the data can't settle.

**What would make it wrong:** If this page answers a quick factual query (e.g. a definition or a simple lookup), low engagement/scroll could be entirely appropriate, which isn't a content failure.

### 7.
**Archetype / action:** engagement_problem_pages → rewrite

**Why it landed here:** High volume (591,696), strong position (2.3), engagement_rate 7.3% — actually the highest of the three engagement-flagged rows shown here, right at the edge of the 30% cutoff isn't close, so it's clearly flagged for a reason.

**Confidence:** Moderate. 7.3% is low in absolute terms, but this is the least severe case in this archetype's sample. It's worth checking whether the 30% engagement threshold is well-calibrated for this site's content type before treating it as equally urgent as #6.

**What would make it wrong:** If 30% is an unrealistic bar for this content category site-wide (i.e. no page here ever clears it), the threshold itself needs recalibrating rather than treating every flagged page as equally broken.

### 8.
**Archetype / action:** engagement_problem_pages → rewrite

**Why it landed here:** High volume (585,712), strong position (2.1), engagement_rate 5.4%.

**Confidence:** Moderate, same caveat as #7 — same client appears three times in this archetype's top rows, worth checking if it's a client-wide GA4 tracking or engagement-definition issue rather than three independent content problems (same pattern flagged earlier for a different client in the original single-rule version).

**What would make it wrong:** Same as #6/#7 and the repeated client here specifically raises the tracking-consistency question.

### 9.
**Archetype / action:** hidden_gems → improve

**Why it landed here:** Good position (8.4), CTR (0.92%) close to expected, but low volume (109 impressions) — under the high-volume bar, so not a champion, but performing well for its size.

**Confidence:** Low. 109 impressions and 1 click is thin; the CTR figure rests on a single click.

**What would make it wrong:** With only 1 click total, this page's "healthy CTR" could flip entirely with one more or one fewer click. There is not strong enough evidence yet to call it a genuine hidden gem versus a low-volume page with an unstable CTR estimate.

### 10.
**Archetype / action:** hidden_gems → improve

**Why it landed here:** Same shape as #9 — 109 impressions, 1 click, decent position (5.5).

**Confidence:** Low, same reasoning as #9.

**What would make it wrong:** Same single-click fragility as #9.

### 11.
**Archetype / action:** hidden_gems → improve

**Why it landed here:** 109 impressions, but 4 clicks (CTR 3.67%, well above expected) at position 9.7.

**Confidence:** Moderate. Still low volume, but 4 clicks out of 109 is a somewhat stronger signal than a single click, and the CTR is meaningfully above the position baseline.

**What would make it wrong:** It's still thin enough that a few more impressions in either direction could change this materially.

### 12.
**Archetype / action:** rising_stars → monitor

**Why it landed here:** Position 11.7 (not yet in the good-position band), but CTR 1.02% clearly beats the ~0.42% expected for positions 4–20, with real volume (60,545 impressions).

**Confidence:** High. Large volume and CTR that clearly outpaces the position baseline is a genuinely interesting positive signal.

**What would make it wrong:** If this keyword/content pairing has an unusually strong brand or intent match (e.g. an exact-match query), the elevated CTR might be intent-specific rather than a sign the page is generally ready to break into top positions.

### 13.
**Archetype / action:** rising_stars → monitor

**Why it landed here:** Position 13.6, CTR 0.60% above the 4–20 baseline (0.42%), volume 54,090.

**Confidence:** Moderate. Real volume, but the CTR outperformance is smaller than #12's — closer to the baseline than a dramatic beat.

**What would make it wrong:** Given the earlier MIXED verdict on CTR-vs-position in the 4–20 range (CTR is nearly flat there), a modest beat like this could be within normal noise for that flat zone rather than a genuine standout.

### 14.
**Archetype / action:** rising_stars → monitor

**Why it landed here:** Position 11.2, CTR 0.48% modestly above the 0.42% baseline, volume 48,538.

**Confidence:** Low-moderate. Same client as #13, and this beat is even smaller — close enough to the baseline that it may not represent real overperformance.

**What would make it wrong:** Same client repetition concern as elsewhere in this review — if this client's CTR baseline differs systematically from the dataset-wide expected-CTR figures (e.g. its audience or brand recognition differs from the rest of the dataset), both #13 and #14 could just reflect a client-level CTR pattern, not page-level "rising star" behavior.

### 15.
**Archetype / action:** stale_visible_pages → refresh

**Why it landed here:** Good position (5.9), solid volume (8,806), but 214 days since update — clearly over the 180-day threshold.

**Confidence:** High. Both the position/volume side and the staleness side are well past their respective thresholds with meaningful sample size behind them.

**What would make it wrong:** If this page's content is evergreen and doesn't need updating (e.g. a stable reference page where freshness isn't a ranking factor for this topic), "stale" by date doesn't necessarily mean "in need of a refresh."

### 16.
**Archetype / action:** stale_visible_pages → refresh

**Why it landed here:** Same client as #15, similar profile — good position (6.1), 215 days stale.

**Confidence:** High, same reasoning as #15.

**What would make it wrong:** Same as #15 and worth checking whether this client simply doesn't update content often as a general practice, in which case "stale" may be this client's norm rather than a red flag.

### 17.
**Archetype / action:** stale_visible_pages → refresh

**Why it landed here:** Third page from the same client in this archetype, 215 days stale, good position (6.1).

**Confidence:** High on the metrics, but the fact that all 3 stale_visible_pages examples come from one client (out of only 27 total in this archetype) suggests this may be a client-specific content-ops pattern (e.g. this client simply refreshes content less often) rather than 27 independently identified problem pages.

**What would make it wrong:** Same as #15/#16.

### 18.
**Archetype / action:** weak_no_demand_pages → prune

**Why it landed here:** Only 29 impressions, 0 clicks, poor position (28.7) — low demand and low performance together.

**Confidence:** Moderate. 29 impressions clears the `MIN_IMPRESSIONS` floor but is still thin.

**What would make it wrong:** A brand-new or highly niche page could show this exact profile temporarily before ever building demand.

### 19.
**Archetype / action:** flag for a data check first, not prune `days_since_last_update = -3` is a negative value, which shouldn't be possible under the intended rule (this is exactly the reporting-lag edge case identified earlier that should route to `INSUFFICIENT_DATA`, but doesn't here because that specific guard wasn't applied in this run).

**Reason code:** weak_no_demand_pages (as currently coded — untrustworthy)

**Confidence:** Low, specifically because of the negative-days issue, separate from the demand numbers themselves.

**What would make it wrong:** If the underlying fix (routing negative-days rows to `INSUFFICIENT_DATA`) is applied, this row would no longer appear in `weak_no_demand_pages` at all. It's a bug artifact, not a genuine archetype match.

### 20.
**Archetype / action:** weak_no_demand_pages → prune

**Why it landed here:** 29 impressions, 0 clicks, very poor position (85.2) — among the clearest "no real demand" profiles in the sample.

**Confidence:** Moderate-high. Position 85 is far enough into the long tail that low demand is credible on its face, not just a threshold artifact.

**What would make it wrong:** A genuinely new page could show this profile briefly before either gaining traction or confirming it truly has no demand; a single snapshot can't distinguish "new and unproven" from "proven and dead."

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

6, 7, 8 are all engagement_problem_pages. Three flagged pages from one client in the sample raises the same concern seen with the earlier single rule version. A client wide GA4 tracking or engagement configuration issue would produce this exact pattern just as easily as three independently under engaging pages would. This should be checked at the client level, confirming GA4 is properly firing across this client's pages generally, before three separate content rewrites are commissioned.

9 and 10 are both hidden_gems resting on exactly 1 click out of 109 impressions. A single click is not stable evidence. One more or one fewer click flips the CTR by nearly half. These pass the MIN_IMPRESSIONS = 10 floor easily, but that floor was calibrated for trusting an impressions based read, similar to the earlier CTR vs position bucket check, not for trusting a specific page's individual CTR estimate. The rule technically fires correctly here, but the underlying evidence is thin. Worth flagging in the write up as a case where meeting the minimum threshold and being trustworthy are not the same thing.

13 and 14 are rising_stars with CTR only modestly above the position baseline, 0.60% and 0.48% versus 0.42% expected. Given the earlier confirmed MIXED finding that CTR barely separates across the 4 to 20 position range, a small beat like this could be noise within that flat zone rather than genuine overperformance. Both rows also come from the same client, raising a second possibility. This client's traffic may simply have a slightly different baseline CTR, from stronger brand recognition or a different audience, than the dataset wide expected CTR figure captures. That would make the beats expectation framing a client level artifact rather than a page level signal.

15, 16, 17 show the same clustering problem for stale_visible_pages. Three of only 27 total rows in this archetype come from a single client. This is likely a client level content ops pattern, this client updates less frequently as a general practice, rather than three independently identified problem pages. The refresh recommendation may be better aimed at the client's update cadence than at these three pages specifically.

19 is the clearest outright bug, not just a low confidence pick. It shows days_since_last_update = -3, which is the reporting lag edge case identified earlier: content_updated_date falling a few days after the fact table's max report_date. The fix for this, routing negative days rows to INSUFFICIENT_DATA instead of letting them flow into the normal staleness logic, was written and confirmed correct, but wasn't actually applied in the code that generated this archetype table. This row shouldn't be trusted as weak_no_demand_pages at all. It's an artifact of an unapplied fix, not a real finding.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

used_cols = ["client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
             "gsc_avg_position", "gsc_data_available", "content_updated_date", "report_date"]

fact_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')").df()["column_name"].tolist()
dim_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()["column_name"].tolist()

print("Fact columns NOT used:", [c for c in fact_cols if c not in used_cols])
print("Dim columns NOT used:", [c for c in dim_cols if c not in used_cols])


Fact columns NOT used: ['client_has_gsc', 'client_has_ga4', 'ga4_data_available', 'gsc_sum_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
Dim columns NOT used: ['keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.